In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import multiprocessing
import os
import pickle

logging.getLogger("pint").setLevel(logging.ERROR)

if os.environ.get("SLURM_CPUS_PER_TASK"):
    cores = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))
else:
    cores = multiprocessing.cpu_count()
print(f"Number of cores: {cores}")

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
from astropy.table import Table
from gpjax.kernels import RBF, Linear, Matern12, Periodic
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.distributions import Normal
from tqdm import tqdm

from gallifrey.gps import KernelSearch, get_trainables, kernel_summary, set_obs_stddev
from gallifrey.inference import (
    log_likelihood_function,
    nuts_warmup,
    run_mcmc,
    create_initial_positions,
)
from gallifrey.util import dict_to_jnp

Number of cores: 8


In [3]:
rng_key = jax.random.PRNGKey(42)

mode = "load"

## LOAD DATA

In [4]:
model_name = "wasp94Ab_gpmodel"

df = Table.read("../data/external/WASP_94Ab.fit").to_pandas()

# time (adjust for simplicity)
t = df["Time"].to_numpy()
t_min = np.amin(t)
t -= t_min

# white lightcurve
white_lc = df["FluxWL"].to_numpy().T
white_lc_err = df["e_FluxWL"].to_numpy().T

# spectroscopic light curves
y = df.iloc[:, 1:-2:2].to_numpy().T
yerr = df.iloc[:, 2:-1:2].to_numpy().T

# mask out transit
mask = np.ones_like(t, dtype=bool)
mask[100:324] = False

# reference parameter from arXiv:2201.02212, last entry is white lc
reference = pd.read_csv("../data/external/WASP_94Ab_reference.csv").set_index(
    df.columns[1::2]
)

planet_period = 0.9501907


num_datasets = len(y)

## KERNEL SEARCH FOR WHITE LC

In [5]:
kernel_library = [
    Linear(),
    RBF(),
    Matern12(),
    Periodic(),
]

In [6]:
if mode == "load":
    with open(
        f"../data/processed/observational_data/gp_models/{model_name}", "rb"
    ) as file:
        model = pickle.load(file)

else:
    tree = KernelSearch(
        kernel_library,
        X=jnp.array(t[mask]),
        y=jnp.array(white_lc[mask]),
        obs_stddev=jnp.amax(white_lc_err[mask]),
        verbosity=1,
        num_threads=cores,
    )

    model = tree.search(
        depth=10,
        n_leafs=3,
        patience=1,
    ).posterior

    if mode == "save":
        with open(
            f"../data/processed/observational_data/gp_models/{model_name}", "wb"
        ) as file:
            pickle.dump(model, file)

summary = kernel_summary(model, to_latex=False)

Fitting Layer 1: 100%|██████████| 4/4 [00:08<00:00,  2.23s/it]


Layer 1 || Current top AICs: [-2955.2424004865434, -2953.973207503252, -2953.7004594024506]


Fitting Layer 2: 100%|██████████| 24/24 [00:40<00:00,  1.69s/it]


Layer 2 || Current top AICs: [-2953.242414347245, -2952.9985580843363, -2951.9732075064844]


Fitting Layer 3: 100%|██████████| 32/32 [01:00<00:00,  1.90s/it]


...

Layer 5 || Current top AICs: [-2957.847211341536, -2957.6738001477624, -2957.6737606932397]


Fitting Layer 6: 100%|██████████| 36/36 [01:12<00:00,  2.00s/it]


Layer 6 || Current top AICs: [-2957.8472113011494, -2957.8471593896593, -2955.847211332584]


Fitting Layer 7: 100%|██████████| 36/36 [01:09<00:00,  1.92s/it]


Layer 7 || Current top AICs: [-2955.847211348185, -2955.8472112578866, -2955.84720736259]
No more improvements found! Terminating early..

Terminated on layer: 7.
Final log likelihood: 1482.923605670768
1482.923605670768
Final number of model parameter: 4
Kernel Summary

Number of Parameter: 4
Kernel Structure: Periodic + Linear • Linear • Linear • Linear
  with obs_stddev = 1.81217e-04 (Trainable : False)

Kernel               Property             Value                Trainable 
--------------------------------------------------------------------------------
Periodic            lengthscale          2.17445e+03          True      

                    variance             1.00140e+00          True      

                    period               3.97816e+03          True      
--------------------------------------------------------------------------------
Linear              variance             7.04769e-03          True      
-----------------------------------------------------------

## DEFINE TRANSIT MODEL

In [ ]:
def transit_model(
    t,
    transit_parameter,
    fixed_parameter,
):
    get_value = (
        lambda key: fixed_parameter[key]
        if key in fixed_parameter
        else transit_parameter[key]
    )

    central = orbits.keplerian.Central(
        mass=get_value("central_mass"),
        radius=get_value("central_radius"),
    )

    orbit = orbits.keplerian.Body(
        central=central,
        period=get_value("period"),
        radius=get_value("planet_radius")
        * central.radius,  # radius in units of stellar radius
        inclination=get_value("inclination"),
        time_transit=get_value("time_transit"),
    )

    u1 = get_value("u1")
    u2 = get_value("u2")
    return LimbDarkLightCurve([u1, u2]).light_curve(orbit, t=t)

## FIT TRANSIT PARAMETER FOR WHITE LC

In [ ]:
fixed_parameter = {"period": planet_period, "u2": reference["u2"]["FluxWL"]}

white_lc_log_likelihood = log_likelihood_function(
    model,
    jit(
        partial(transit_model, fixed_parameter=fixed_parameter)
    ),  # partial returns fixed_parameter fixed
    t,
    white_lc,
    mask,
    fix_gp=False,
    compile=True,
    negative=True,
)

x0 = {
    "gp_parameter": get_trainables(model, unconstrain=True),
    "lc_parameter": {
        "planet_radius": 0.11,
        "u1": 0.53,
        "central_mass": 1.45,
        "central_radius": 1.653,
        "inclination": jnp.deg2rad(89.3),
        "time_transit": 0.18,
    },
}
white_lc_solve = ScipyMinimize(
    fun=white_lc_log_likelihood,
    method="l-bfgs-b",
).run(x0)
white_lc_parameter = white_lc_solve.params["lc_parameter"]

## DEFINE LIKELIHOOD, PRIOR, POSTERIOR

In [ ]:
def get_logprob(
    gp_model,
    y,
    yerr,
    u1,
    u2,
    initial_position=None,
    fix_gp=False,
):
    if initial_position is None:
        initial_position = {
            "gp_parameter": get_trainables(gp_model, unconstrain=True),
            "lc_parameter": {"planet_radius": 0.10, "u1": u1},
        }

    # define transit light curve model
    fixed_parameter = {
        param: white_lc_parameter[param]
        for param in (
            "central_mass",
            "central_radius",
            "inclination",
            "time_transit",
        )
    }
    fixed_parameter["u2"] = u2
    fixed_parameter["period"] = planet_period
    lc_model = jit(partial(transit_model, fixed_parameter=fixed_parameter))

    # update observational uncertainty for lc models
    gp_model = set_obs_stddev(gp_model, jnp.amax(yerr[mask]))

    # get log likelihood function
    log_likelihood = log_likelihood_function(
        gp_model,
        lc_model,
        t,
        y,
        mask,
        fix_gp=fix_gp,
        compile=True,
    )

    # define priors
    param_priors = {
        "gp_parameter": Normal(
            loc=initial_position["gp_parameter"],
            scale=0.1 * jnp.abs(initial_position["gp_parameter"]),
        ),
        "lc_parameter": Normal(
            loc=dict_to_jnp(initial_position["lc_parameter"]),
            scale=[0.2, 0.05],
        ),
    }

    @jit
    def log_priors(params):
        gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
        lc_log_priors = param_priors["lc_parameter"].log_prob(
            dict_to_jnp(params["lc_parameter"])
        )
        return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)

    # put it all together to full posterior probability function
    @jit
    def log_probability(params):
        return log_likelihood(params) + log_priors(params)

    return log_probability, initial_position

## PERFORM FITS

In [ ]:
parameter_solutions = []
for i in tqdm(range(num_datasets)):
    log_probability, initial_position = get_logprob(
        model,
        y[i],
        yerr[i],
        reference["u1_linear"].iloc[i],
        reference["u2"].iloc[i],
    )

    solve = ScipyMinimize(
        fun=jit(lambda par: -log_probability(par)), method="l-bfgs-b"
    ).run(initial_position)
    parameter_solutions.append(solve.params)

## RUN MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 150
num_samples = 80
num_chains = cores

fix_gp = False

In [ ]:
rng_key, warmup_key = jax.random.split(rng_key, 2)

if mode != "load":
    # run nuts adaption on white lc
    log_probability, initial_position = get_logprob(
        model,
        white_lc,
        white_lc_err,
        reference["u1_linear"]["FluxWL"],
        reference["u2"]["FluxWL"],
        fix_gp=fix_gp,
    )

    state, nuts_parameters = nuts_warmup(
        warmup_key,
        log_probability,
        initial_position,
        num_steps=num_adapt,
    )

In [ ]:
chains = {"gp_parameter": [], "lc_parameter": []}

if mode == "load":
    chains = np.load(
        f"../data/processed/observational_data/mcmc_chains/{model_name}_parameter.npz"
    )

else:
    for i in tqdm(range(num_datasets)):
        log_probability, initial_position = get_logprob(
            model,
            y[i],
            yerr[i],
            reference["u1_linear"].iloc[i],
            reference["u2"].iloc[i],
            initial_position=parameter_solutions[i],
            fix_gp=fix_gp,
        )

        # define initial positions and add scatter
        rng_key, gp_scatter_key = jax.random.split(rng_key, 2)

        initial_positions = {}
        initial_positions["gp_parameter"] = create_initial_positions(
            initial_position["gp_parameter"],
            num=num_chains,
            sigma=0.05,
            key=gp_scatter_key,
        )
        initial_positions["lc_parameter"] = {}
        for name, value in initial_position["lc_parameter"].items():
            rng_key, scatter_key = jax.random.split(rng_key, 2)
            initial_positions["lc_parameter"][name] = create_initial_positions(
                value,
                num=num_chains,
                sigma=0.05,
                key=scatter_key,
            )

        # run mcmc
        rng_key, sample_key = jax.random.split(rng_key, 2)

        final_state, state_history, info_history = run_mcmc(
            sample_key,
            log_probability,
            nuts_parameters,
            initial_positions,
            num_steps=num_samples,
        )

        for par in ["gp_parameter", "lc_parameter"]:
            chain = dict_to_jnp(state_history.position[par])
            chains[par].append(chain)

    if mode == "save":
        np.savez(
            f"../data/processed/observational_data/mcmc_chains/{model_name}_parameter.npz",
            **chains,
        )